# 06 · Backtesting en train

No registra declaraciones nuevas: exige que `[forecast]` se haya declarado antes de la réplica y que no haya cambiado. Solo carga días de train y no lee el EDA ni los artefactos de la réplica.

13 orígenes cada 28 días en Y4 (R16): cada candidato se ajusta una vez con los días ≤ d_1213 y los folds solo añaden observaciones. La participación usa `zero_thresholds` congelado: el arranque y las rachas internas ≥ umbral quedan NaN, y cada historia se enmascara solo con los días previos a su origen. El RMSSE se mide sobre las unidades reales.

In [ ]:
import os
import sys
from pathlib import Path

# Raíz del código: el notebook corre desde notebooks/. PROJECT_ROOT permite apuntar a otra carpeta
# de datos (por ejemplo, una copia sintética para validar el flujo).
CODE_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(CODE_ROOT))
os.chdir(os.environ.get("PROJECT_ROOT", CODE_ROOT))

import numpy as np
import pandas as pd

from src.config import load_validated_config

pd.set_option("display.max_columns", 30)
cfg = load_validated_config()
print("versión de la config:", cfg["version"])

In [ ]:
from src.freeze import load_frozen

zt = load_frozen("zero_thresholds")
zt.groupby(["source", "reason"], dropna=False).size().rename("artículos")

In [ ]:
from src.forecasting import fold_starts, run_backtesting_stage

bt = run_backtesting_stage(cfg)
bt["scores"].groupby("candidate_id")["rmsse_mean"].mean().sort_values().head(15)

RMSSE medio por fold y mes de inicio del fold, para ver el ciclo anual (pico de verano en el nb 02, 9.3–9.4).

In [ ]:
from src.data import load_calendar

cal = load_calendar(cfg)
starts = pd.Series(fold_starts(cfg), index=range(1, len(fold_starts(cfg)) + 1), name="start_d")
month = pd.to_datetime(cal.set_index(cal["d"].str[2:].astype(int)).loc[starts, "date"]).dt.strftime("%Y-%m")
by_fold = bt["scores"].groupby(["fold", "candidate_id"])["rmsse_mean"].mean().unstack()
best = bt["scores"].groupby("candidate_id")["rmsse_mean"].mean().nsmallest(5).index
table = by_fold[best].set_axis(month.to_numpy(), axis=0).rename_axis("mes de inicio")
table.style.format("{:.3f}").background_gradient(axis=None, cmap="Reds")

In [ ]:
pd.Series([m["candidate_id"] for m in bt["models"]]).value_counts()

Fallos de ajuste registrados (artículo, candidato, mensaje).

In [ ]:
failures = bt["failures"]
print(len(failures), "fallos")
failures.groupby(["stage"]).size() if len(failures) else failures